In [19]:
# 03_read_date_LLM_vs_labels.ipynb
# Given the annotated dataset with the expected date labels and the values in response from the LLM, it calculates the correctness of the returned responses with respect to the labels.  
# Use https://openai.com/

In [20]:
# Force to reload extrernal modules every new cell execution
%reload_ext autoreload
%autoreload 2

In [21]:
### IMPORT ###
from pathlib import Path
import csv
from datetime import datetime
import pandas as pd

In [22]:
### LOCAL IMPORT ###
from config import config_reader
from utilities import read_csv_data_to_df, convert_dmy_to_ymd, left_join_df, calculate_accuracy

In [26]:
### GLOBALS ###
yaml_config = config_reader.config_read_yaml("config.yml", "config")
# print(yaml_config) # debug
data_dir = str(yaml_config["DATA_DIR"])
csv_sep = str(yaml_config["CSV_SEP"])
sample_size = int(yaml_config["TEST_SAMPLE"]) 

# INPUT
dic_lan = {"DE":1, "ES":0, "FR":0, "IT":0, "PT":0} # <-- INPUT: set 1 for the desired language; set only one language at time
bid_file_text_date_label = str(yaml_config["FILE_BID_TEXT_DATE_LABEL"]) # the label file based 

## FUNCTIONS

## MAIN

In [27]:
### MAIN ###
print()
print("*** PROGRAM START ***")
print()

start_time = datetime.now().replace(microsecond=0)
print("Start process:", str(start_time))
print()


*** PROGRAM START ***

Start process: 2024-09-16 10:13:20



In [28]:
print(">> Settings")
lang_code = None
key_with_value_1 = [key for key, value in dic_lan.items() if value == 1]
if key_with_value_1: 
    lang_code = key_with_value_1[0]
print("Desired language code:", lang_code)

# input
bid_file_text_lang = bid_file_text_date_label.replace("LANG", lang_code)
path_bid_text_lang = Path(data_dir) / bid_file_text_lang
print("File input:", path_bid_text_lang)

>> Settings
Desired language code: DE
File input: data/bid_opening_text_DE_date_llm_OAI_label.xlsx


In [29]:
# Reading CSV file text and dates to be extracted by LLM
print(">> Reading CSV input file")
print("Path:", str(path_bid_text_lang))
dic_type = {"file_name":object, "case_id":object, "text":object, "text_date":object, "text_date_llm":object, "label":object}
df_bid_text = pd.read_excel(path_bid_text_lang, dtype=dic_type)
df_bid_text_len = len(df_bid_text) # all the rows in the dataframe
print("Rows in dataframe:", df_bid_text_len)
print("Columns in dataframe:", df_bid_text.columns)

>> Reading CSV input file
Path: data/bid_opening_text_DE_date_llm_OAI_label.xlsx
Rows in dataframe: 150012
Columns in dataframe: Index(['file_name', 'case_id', 'text', 'text_date', 'text_date_llm', 'label'], dtype='object')


In [14]:
df_bid_text.head(5)

,file_name,case_id,text_date,text_date_llm,label
0,2016-OJS001-00000108-it-ts.pdf,2016108,Data: 16.2.2016 - 9:00,2016-02-16,2016-02-16
1,2016-OJS001-00000236-it-ts.pdf,2016236,Data: 3.3.2016 - 10:00,2016-03-03,2016-03-03 00:00:00
2,2016-OJS001-00000277-it-ts.pdf,2016277,Persone ammesse ad assistere all'apertura dell...,0,0
3,2016-OJS001-00000307-it-ts.pdf,2016307,Data: 24.2.2016 - 10:00,2016-02-24,2016-02-24
4,2016-OJS001-00000320-it-ts.pdf,2016320,Data: 17.3.2016 - 10:00,2016-03-17,2016-03-17


In [30]:
# Count the rows with NaN in the 'label' column
nan_count = df_bid_text['label'].isna().sum()

# Count the rows with an empty string in the 'label' column
empty_string_count = (df_bid_text['label'] == '').sum()

# Total sum of empty or NaN values
total_empty_or_nan = nan_count + empty_string_count

print(f"Rows with NaN: {nan_count}")
print(f"Rows with an empty string: {empty_string_count}")
print(f"Total rows that are empty or NaN: {total_empty_or_nan}")

Rows with NaN: 139512
Rows with an empty string: 0
Total rows that are empty or NaN: 139512


In [31]:
print("Considering only non-empty rows with labels")
df_bid_text = df_bid_text[(df_bid_text['label'].notna()) & (df_bid_text['label'] != '')]
df_bid_text_len_clean = len(df_bid_text) # the rows in the dataframe with label non empty
print(f"Considering {df_bid_text_len_clean} out of {df_bid_text_len}")

Considering only non-empty rows with labels
Considering 10500 out of 150012


In [32]:
# Convert 'text_date_llm' column to string to ensure proper comparisons
df_bid_text['text_date_llm'] = df_bid_text['text_date_llm'].astype(str)

# Clean the "label" column to match the format of "text_date_llm" by removing the time component
df_bid_text['label_clean'] = df_bid_text['label'].astype(str).str[:10]

# Step 1: Count how many rows have 'text_date_llm' equal to '0'
zero_rows_count = len(df_bid_text[df_bid_text['text_date_llm'] == '0'])
zero_rows_percentage = (zero_rows_count / len(df_bid_text)) * 100

# Step 2: Exclude rows where 'text_date_llm' is '0'
valid_rows = df_bid_text[df_bid_text['text_date_llm'] != '0']

# Step 3: Calculate the total number of rows after exclusion
total_valid_rows = len(valid_rows)

# Step 4: Compare 'text_date_llm' and 'label_clean' and count matches
matches = valid_rows[valid_rows['text_date_llm'] == valid_rows['label_clean']]
matching_rows_count = len(matches)

# Step 5: Calculate the percentage of matching rows
matching_percentage = (matching_rows_count / total_valid_rows) * 100

# Output the results
print(f"Rows with 'text_date_llm' = '0': {zero_rows_count} ({zero_rows_percentage:.2f}%)")
print(f"Matching rows (excluding '0'): {matching_rows_count} ({matching_percentage:.2f}%)")

Rows with 'text_date_llm' = '0': 2884 (27.47%)
Matching rows (excluding '0'): 7616 (100.00%)


In [33]:
# program end
end_time = datetime.now().replace(microsecond=0)
delta_time = end_time - start_time

print()
print("End process:", end_time)
print("Time to finish:", delta_time)
print()

print()
print("*** PROGRAM END ***")
print()


End process: 2024-09-16 10:18:02
Time to finish: 0:04:42


*** PROGRAM END ***

